## Homework 9: Text Classification with Fine-Tuned BERT

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [12]:
# Install once per new Colab runtime
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate

In [13]:

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

### Here is where you can set global hyperparameters for this homework

In [14]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 128
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

In [15]:
# ---- Load IMDb (raw), join train+test ----
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

# ---- Build DS with explicit features (label=ClassLabel) ----
features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

# ---- Optional: take a stratified subset of the FULL dataset ----
if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

# ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
# First: 80/20 train+val pool / test
splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
# Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

# ---- Numpy arrays for Keras fit/predict ----
X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

# ---- Quick summary ----
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()
print(f"Pool after SUBSET_FRAC={SUBSET_FRAC}: {len(ds_pool)} (of {len(all_ds)})")
print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))


Pool after SUBSET_FRAC=0.25: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])


### Build and train a baseline Distil-Bert Text Classifier

In [16]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 104s 226ms/step - acc: 0.7824 - loss: 0.4530 - val_acc: 0.8376 - val_loss: 0.3448
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - acc: 0.8787 - loss: 0.2897 - val_acc: 0.8584 - val_loss: 0.3400
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - acc: 0.9160 - loss: 0.2207 - val_acc: 0.8592 - val_loss: 0.3555

Validation acc (best epoch): 0.858

Test accuracy: 0.855   Test F1: 0.851

Confusion matrix:
 [[1098  152]
 [ 211 1039]]

Elapsed time: 00:02:24


# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [17]:
# Run these experiments in Colab. The local macOS TensorFlow stack is often unstable for this homework.
import gc
import tensorflow as tf

MODEL_SPECS = {
    "distilbert": {
        "label": "DistilBERT",
        "preprocessor_cls": kh.models.DistilBertTextClassifierPreprocessor,
        "classifier_cls": kh.models.DistilBertTextClassifier,
        "preset": "distil_bert_base_en_uncased",
    },
    "bert_base": {
        "label": "BERT-base",
        "preprocessor_cls": kh.models.BertTextClassifierPreprocessor,
        "classifier_cls": kh.models.BertTextClassifier,
        "preset": "bert_base_en_uncased",
    },
}


def prepare_subset_pool(full_ds, subset_frac, seed=SEED):
    if 0.0 < subset_frac < 1.0:
        sub = full_ds.train_test_split(train_size=subset_frac, seed=seed, stratify_by_column="label")
        return sub["train"]
    return full_ds


def split_pool(pool_ds, seed=SEED):
    splits = pool_ds.train_test_split(test_size=0.20, seed=seed, stratify_by_column="label")
    train_val_pool, test_ds = splits["train"], splits["test"]
    splits2 = train_val_pool.train_test_split(test_size=0.125, seed=seed, stratify_by_column="label")
    train_ds, val_ds = splits2["train"], splits2["test"]

    X_tr = np.array(train_ds["text"], dtype=object)
    y_tr = np.array(train_ds["label"], dtype="int32")
    X_va = np.array(val_ds["text"], dtype=object)
    y_va = np.array(val_ds["label"], dtype="int32")
    X_te = np.array(test_ds["text"], dtype=object)
    y_te = np.array(test_ds["label"], dtype="int32")
    return (X_tr, y_tr), (X_va, y_va), (X_te, y_te)


def binary_f1_np(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


def build_classifier(model_name, max_len, lr):
    spec = MODEL_SPECS[model_name]
    preproc = spec["preprocessor_cls"].from_preset(spec["preset"], sequence_length=max_len)
    model = spec["classifier_cls"].from_preset(spec["preset"], num_classes=2, preprocessor=preproc)
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    return model


def run_experiment(model_name, max_len, lr, batch_size, subset_frac, epochs=EPOCHS, seed=SEED, verbose=2):
    keras.utils.set_random_seed(seed)
    keras.backend.clear_session()
    gc.collect()

    pool_ds = prepare_subset_pool(all_ds, subset_frac, seed=seed)
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = split_pool(pool_ds, seed=seed)
    model = build_classifier(model_name, max_len=max_len, lr=lr)
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]

    start = time.time()
    history = model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose,
    )
    elapsed = time.time() - start

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)
    best_idx = int(np.argmin(history.history["val_loss"]))

    result = {
        "model_name": model_name,
        "model_label": MODEL_SPECS[model_name]["label"],
        "subset_frac": float(subset_frac),
        "max_len": int(max_len),
        "lr": float(lr),
        "batch": int(batch_size),
        "epochs_ran": len(history.history["loss"]),
        "best_epoch": best_idx + 1,
        "best_val_loss": float(history.history["val_loss"][best_idx]),
        "best_val_acc": float(history.history["val_acc"][best_idx]),
        "test_acc": float(np.mean(y_pred == y_te)),
        "test_f1": float(binary_f1_np(y_te, y_pred)),
        "elapsed_sec": float(elapsed),
        "train_size": len(X_tr),
        "val_size": len(X_va),
        "test_size": len(X_te),
    }
    return result, history


def print_results_table(results, title):
    print(title)
    header = (
        f"{'model':<11} {'frac':>5} {'len':>5} {'lr':>10} {'batch':>6} {'best_ep':>7} "
        f"{'val_loss':>9} {'val_acc':>8} {'test_acc':>9} {'test_f1':>8} {'time_m':>8}"
    )
    print(header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['model_label']:<11} {r['subset_frac']:>5.2f} {r['max_len']:>5} "
            f"{r['lr']:>10.2e} {r['batch']:>6} {r['best_epoch']:>7} "
            f"{r['best_val_loss']:>9.4f} {r['best_val_acc']:>8.4f} "
            f"{r['test_acc']:>9.4f} {r['test_f1']:>8.4f} {r['elapsed_sec'] / 60:>8.2f}"
        )


P1_CONFIGS = [
    {"max_len": 128, "lr": 2e-5, "batch_size": 32},
    {"max_len": 128, "lr": 1e-5, "batch_size": 32},
    {"max_len": 256, "lr": 1e-5, "batch_size": 16},
    {"max_len": 256, "lr": 7.5e-6, "batch_size": 16},
    {"max_len": 512, "lr": 7.5e-6, "batch_size": 8},
    {"max_len": 512, "lr": 5e-6, "batch_size": 8},
]

results_p1 = []
for cfg in P1_CONFIGS:
    print(
        f"\nRunning Problem 1 config: MAX_LEN={cfg['max_len']} "
        f"LR={cfg['lr']:.2e} BATCH={cfg['batch_size']}"
    )
    try:
        result, _ = run_experiment("distilbert", subset_frac=0.25, **cfg)
    except tf.errors.ResourceExhaustedError as exc:
        print("Out of memory for this configuration.")
        print("Reduce the batch size exactly as suggested in the homework prompt, then rerun this cell.")
        raise exc
    results_p1.append(result)

results_p1 = sorted(results_p1, key=lambda r: (r["best_val_loss"], -r["best_val_acc"]))
print_results_table(results_p1, title="\nProblem 1 summary")
best_p1 = min(results_p1, key=lambda r: r["best_val_loss"])
print("\nBest Problem 1 config:")
print(best_p1)

best_by_len = {}
for max_len in sorted({r["max_len"] for r in results_p1}):
    runs = [r for r in results_p1 if r["max_len"] == max_len]
    best_by_len[max_len] = max(runs, key=lambda r: r["best_val_acc"])

max_lr_gap = max(
    max(r["best_val_acc"] for r in results_p1 if r["max_len"] == max_len)
    - min(r["best_val_acc"] for r in results_p1 if r["max_len"] == max_len)
    for max_len in {r["max_len"] for r in results_p1}
)
len_summary = ", ".join(
    f"{max_len}: {best_by_len[max_len]['best_val_acc']:.4f}" for max_len in sorted(best_by_len)
)
avg_by_len = {
    max_len: float(np.mean([r["best_val_acc"] for r in results_p1 if r["max_len"] == max_len]))
    for max_len in sorted(best_by_len)
}
monotonic_increase = avg_by_len[128] <= avg_by_len[256] <= avg_by_len[512]
a1b_answer = (
    "More context "
    + ("generally helped across this sweep. " if monotonic_increase else "did not consistently help across this sweep. ")
    + f"The best validation accuracy at each context length was {len_summary}. "
    + f"Changing the learning rate within a fixed context length moved validation accuracy by at most about {max_lr_gap:.4f}, "
    + "so LR mattered, but its effect was modest compared with the overall pattern across runs."
)
print("\nSuggested answer for a1b:\n")
print(a1b_answer)


### Graded Questions

In [18]:
if "best_p1" not in globals():
    raise RuntimeError("Run the Problem 1 experiment cell first.")

a1a = float(best_p1["best_val_acc"])


In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

#### Question a1b:

* Does **more context** (128 → 256 → 512) consistently help?
* How much effect did the learning rate have on the validation accuracy?

#### Your Answer Here:

Run the Problem 1 sweep cell above and use the generated `a1b_answer` text. It summarizes the actual trend from your six runs instead of forcing you to guess before the experiments finish.


## Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [20]:
if "best_p1" not in globals():
    raise RuntimeError("Run Problem 1 before starting Problem 2.")

P2_FRACTIONS = [0.25, 0.50, 0.75, 1.00]
results_p2 = []
for frac in P2_FRACTIONS:
    print(f"\nRunning Problem 2 with SUBSET_FRAC={frac:.2f}")
    result, _ = run_experiment(
        "distilbert",
        max_len=best_p1["max_len"],
        lr=best_p1["lr"],
        batch_size=best_p1["batch"],
        subset_frac=frac,
    )
    results_p2.append(result)

results_p2 = sorted(results_p2, key=lambda r: r["subset_frac"])
print_results_table(results_p2, title="\nProblem 2 summary")
best_p2 = min(results_p2, key=lambda r: r["best_val_loss"])
print("\nBest Problem 2 config:")
print(best_p2)

p2_by_frac = {r["subset_frac"]: r for r in results_p2}
small = p2_by_frac[0.25]
full = p2_by_frac[1.00]
acc_gain = full["best_val_acc"] - small["best_val_acc"]
time_ratio = full["elapsed_sec"] / small["elapsed_sec"]
best_frac = max(results_p2, key=lambda r: r["best_val_acc"])["subset_frac"]

if acc_gain >= 0.01:
    justification = "The extra data looks worthwhile if you want the best possible score."
else:
    justification = (
        "Because the gain is small relative to normal validation noise at two decimal places, "
        "the full dataset is hard to justify for routine experimentation."
    )

a2b_answer = (
    f"As dataset size increased, validation accuracy went from {small['best_val_acc']:.4f} at 25% of the data "
    f"to {full['best_val_acc']:.4f} at 100%, while training time grew from {small['elapsed_sec'] / 60:.2f} "
    f"to {full['elapsed_sec'] / 60:.2f} minutes. "
    f"That is a gain of {acc_gain:.4f} in validation accuracy for about {time_ratio:.2f}x more compute. "
    + justification
    + f" In these runs, the best validation accuracy came from SUBSET_FRAC={best_frac:.2f}."
)
print("\nSuggested answer for a2b:\n")
print(a2b_answer)


### Graded Questions

In [21]:
if "best_p2" not in globals():
    raise RuntimeError("Run the Problem 2 experiment cell first.")

a2a = float(best_p2["best_val_acc"])


In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

#### Question a2b:

Summarize what you observed as dataset size increased. Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset? What trade-offs between accuracy and computation time did you notice?

#### Your Answer Here:

Run the Problem 2 cell above and use the generated `a2b_answer` text. It plugs in the observed validation accuracies and elapsed times from your own runs.


# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [23]:
if "best_p1" not in globals() or "best_p2" not in globals():
    raise RuntimeError("Run Problems 1 and 2 before starting Problem 3.")

P3_MODELS = ["distilbert", "bert_base"]
results_p3 = []
for model_name in P3_MODELS:
    print(f"\nRunning Problem 3 with {MODEL_SPECS[model_name]['label']}")
    result, _ = run_experiment(
        model_name,
        max_len=best_p1["max_len"],
        lr=best_p1["lr"],
        batch_size=best_p1["batch"],
        subset_frac=best_p2["subset_frac"],
    )
    result["sec_per_epoch"] = result["elapsed_sec"] / result["epochs_ran"]
    results_p3.append(result)

results_p3 = sorted(results_p3, key=lambda r: (r["best_val_loss"], -r["test_acc"]))
print_results_table(results_p3, title="\nProblem 3 summary")
best_p3 = min(results_p3, key=lambda r: r["best_val_loss"])
fastest_p3 = min(results_p3, key=lambda r: r["sec_per_epoch"])
best_acc_p3 = max(results_p3, key=lambda r: r["test_acc"])
best_f1_p3 = max(results_p3, key=lambda r: r["test_f1"])

if best_acc_p3["model_label"] == fastest_p3["model_label"]:
    overall_choice = fastest_p3["model_label"]
    overall_reason = "it delivered the strongest accuracy while also being the fastest option in this comparison"
else:
    acc_gap = best_acc_p3["test_acc"] - fastest_p3["test_acc"]
    if acc_gap < 0.01:
        overall_choice = fastest_p3["model_label"]
        overall_reason = "its accuracy was very close to the best model, but it trained noticeably faster"
    else:
        overall_choice = best_acc_p3["model_label"]
        overall_reason = "the accuracy gain was large enough to justify the extra compute"

a3b_answer = (
    f"{best_acc_p3['model_label']} had the best test accuracy ({best_acc_p3['test_acc']:.4f}), "
    f"and {best_f1_p3['model_label']} had the best F1 ({best_f1_p3['test_f1']:.4f}). "
    f"{fastest_p3['model_label']} was the fastest model at about {fastest_p3['sec_per_epoch']:.2f} seconds per epoch. "
    f"Given limited development time or compute, I would pick {overall_choice} because {overall_reason}."
)
print("\nBest Problem 3 model:")
print(best_p3)
print("\nSuggested answer for a3b:\n")
print(a3b_answer)


### Graded Questions

In [24]:
if "best_p3" not in globals():
    raise RuntimeError("Run the Problem 3 experiment cell first.")

a3a = float(best_p3["best_val_acc"])


In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

#### Question a3b:

**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?

#### Your Answer Here:

Run the Problem 3 cell above and use the generated `a3b_answer` text. It compares the two models using the actual accuracy, F1, and speed numbers from your run.
